# Step 5: Ensemble Evaluation (LC-Protonets + CoMC)

## Goals
1. Load the LC-Protonets model trained in Step 3.
2. Load the CoMC classifier trained in Step 4.
3. Combine predictions as p_final = alpha * p_proto + (1 - alpha) * p_comc.
4. Evaluate the ensemble on 16 novel classes.

## Ensemble Strategy
- LC-Protonets uses support and query sets with metric learning.
- CoMC uses query images, CLIP image features, and cross-modal mapping.
- The ensemble weight is configurable.

## Metrics
Report multi-label classification metrics for both component models and the ensemble.


## 5.1 Import Required Libraries


In [ ]:
import os
import json
import numpy as np
import random
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

import clip

from sklearn.metrics import (
    average_precision_score,
    precision_recall_fscore_support,
    f1_score
)

# Independent full-pipeline seed supplied by run_multiseed_experiment.py
EXPERIMENT_SEED = int(os.environ.get('EXPERIMENT_SEED', '42'))

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    strict_determinism = os.environ.get('STRICT_DETERMINISM', '0') == '1'
    torch.backends.cudnn.deterministic = strict_determinism
    torch.backends.cudnn.benchmark = not strict_determinism
    if strict_determinism:
        torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(EXPERIMENT_SEED)
print(f'Experiment seed: {EXPERIMENT_SEED}')
print(f'Strict CUDA determinism: {os.environ.get("STRICT_DETERMINISM", "0")}')

PROGRESS_FILE = os.environ.get('RUN_PROGRESS_FILE')

def update_run_progress(phase, current, total, **details):
    if not PROGRESS_FILE:
        return
    payload = {
        'seed': EXPERIMENT_SEED,
        'phase': phase,
        'current': int(current),
        'total': int(total),
        'details': details
    }
    os.makedirs(os.path.dirname(PROGRESS_FILE), exist_ok=True)
    temporary_file = PROGRESS_FILE + '.tmp'
    with open(temporary_file, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle)
    os.replace(temporary_file, PROGRESS_FILE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 5.2 Configure Paths and Hyperparameters


In [ ]:
DATASET_ROOT = os.environ.get("COCO_ROOT", "coco2014")
TRAIN2014_DIR = os.path.join(DATASET_ROOT, "train2014")
VAL2014_DIR = os.path.join(DATASET_ROOT, "val2014")

DATA_DIR = os.environ.get("DEPF_DATA_DIR", "coco2014_data")
SPLIT_FILE = os.path.join(DATA_DIR, "coco_mutually_exclusive_split.json")
MAPPING_FILE = os.path.join(DATA_DIR, "category_mapping.json")

# ==================== Seed-isolated paths ====================
MULTISEED_ROOT = os.environ.get(
    'MULTISEED_ROOT',
    'coco2014_multiseed_k5'
)
RUN_DIR = os.path.join(MULTISEED_ROOT, f'seed_{EXPERIMENT_SEED}')

LC_PROTONETS_CKPT = os.path.join(
    RUN_DIR, 'lc_protonets', 'checkpoints',
    'lc_protonets_52way_5shot_dim2048_finetuneTrue_euclidean_bias14.7.pth'
)

COMC_CLASSIFIER_CKPT = os.path.join(RUN_DIR, 'comc', 'comc_classifier_best.pth')

OUTPUT_DIR = os.path.join(RUN_DIR, 'evaluation')
os.makedirs(OUTPUT_DIR, exist_ok=True)

ALPHA = 0.1
N_WAY = 16
K_SHOT = 5
N_QUERY = 4
N_EPISODES = 200
COMC_ALPHA = 0.7
COMC_TEMPERATURE = 0.07
CLIP_MODEL = "ViT-B/32"

print("="*70)
print("ENSEMBLE EVALUATION CONFIGURATION")
print("="*70)
print(f"\nDataset:")
print(f"  Novel classes: {N_WAY}")
print(f"  k_shot: {K_SHOT}")
print(f"  n_query: {N_QUERY}")
print(f"  n_episodes: {N_EPISODES}")
print(f"\nEnsemble:")
print(f"  Alpha (fusion weight): {ALPHA}")
print(f"  Formula: p_final = {ALPHA}*p_proto + {1-ALPHA}*p_comc")
print(f"\nModels:")
print(f"  LC-Protonets: {LC_PROTONETS_CKPT}")
print(f"  CoMC: {COMC_CLASSIFIER_CKPT}")
print(f"\nOutput directory: {OUTPUT_DIR}")
print("="*70)

## 5.3 Load Dataset Splits and Class Mappings


In [ ]:
print("Loading dataset split...")
with open(SPLIT_FILE, 'r') as f:
    data_split = json.load(f)

with open(MAPPING_FILE, 'r') as f:
    category_mapping = json.load(f)

cat_name_to_id = category_mapping['cat_name_to_id']
cat_id_to_name = category_mapping['cat_id_to_name']

novel_samples = data_split['novel_samples']
novel_class_names = data_split['metadata']['novel_classes']
novel_cat_ids = [cat_name_to_id[name] for name in novel_class_names]

print(f" Dataset loaded!")
print(f"  Novel classes: {len(novel_class_names)}")
print(f"  Novel samples: {len(novel_samples)}")
print(f"\nNovel classes:")
for i, (name, cat_id) in enumerate(zip(novel_class_names, novel_cat_ids)):
    print(f"  {i}: {name} (ID: {cat_id})")

## 5.4 Define a Dataset with Two Preprocessing Pipelines


In [ ]:
class DualPreprocessCOCODataset(Dataset):
    """Load COCO images with separate preprocessing for LC-Protonets and CLIP."""
    
    def __init__(
        self,
        samples: List[Dict],
        train_image_dir: str,
        val_image_dir: str,
        lc_transform=None,        comc_transform=None,        cat_id_to_name: Dict[int, str] = None
    ):
        self.samples = samples
        self.train_image_dir = train_image_dir
        self.val_image_dir = val_image_dir
        self.lc_transform = lc_transform
        self.comc_transform = comc_transform
        self.cat_id_to_name = cat_id_to_name or {}
    
    def __len__(self):
        return len(self.samples)
    
    def _get_image_path(self, sample: Dict) -> str:
        """Find an image in the configured COCO image directories."""
        filename = sample.get('filename', f"{sample['image_id']:012d}.jpg")
        
        val_path = os.path.join(self.val_image_dir, filename)
        if os.path.exists(val_path):
            return val_path
        
        train_path = os.path.join(self.train_image_dir, filename)
        if os.path.exists(train_path):
            return train_path
        
        raise FileNotFoundError(f"Image not found: {filename}")
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        image_path = self._get_image_path(sample)
        image = Image.open(image_path).convert('RGB')
        
        if self.lc_transform:
            image_lc = self.lc_transform(image)
        else:
            image_lc = image
        
        if self.comc_transform:
            image_comc = self.comc_transform(image)
        else:
            image_comc = image
        
        labels = torch.tensor(sample['category_ids'], dtype=torch.long)
        
        return {
            'image_lc': image_lc,
            'image_comc': image_comc,
            'labels': labels,
            'image_id': sample['image_id'],
            'category_ids': sample['category_ids']
        }

print("DualPreprocessCOCODataset class defined!")

## 5.5 Define LC-Protonets and CoMC Components


In [ ]:
import torchvision.models as models

# ==================== Feature Extractor (from Step 3) ====================
class FeatureExtractor(nn.Module):
    def __init__(self, pretrained=True, feature_dim=2048, finetune=True):
        super(FeatureExtractor, self).__init__()
        resnet = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        
        if feature_dim != 2048:
            self.projection = nn.Sequential(
                nn.Flatten(),
                nn.Linear(2048, feature_dim),
                nn.ReLU(inplace=True)
            )
        else:
            self.projection = nn.Flatten()
        
        self.finetune = finetune
        if not finetune:
            for param in self.features.parameters():
                param.requires_grad = False
    
    def forward(self, x):
        x = self.features(x)
        return self.projection(x)

# ==================== LC-Protonets (from Step 3) ====================
class LCProtonets(nn.Module):
    def __init__(self, feature_extractor, distance_metric='euclidean', init_bias=14.7):
        super(LCProtonets, self).__init__()
        self.feature_extractor = feature_extractor
        self.distance_metric = distance_metric
        self.bias = nn.Parameter(torch.tensor(init_bias))
    
    def compute_prototypes(self, support_features, support_labels):
        from collections import defaultdict
        label_combinations = [tuple(sorted(labels)) for labels in support_labels]
        combination_to_features = defaultdict(list)
        for features, combo in zip(support_features, label_combinations):
            combination_to_features[combo].append(features)
        
        prototypes = {}
        for combo, feature_list in combination_to_features.items():
            features_stack = torch.stack(feature_list)
            prototypes[combo] = features_stack.mean(dim=0)
        return prototypes
    
    def compute_distances(self, query_features, prototypes):
        distances = {}
        for combo, prototype in prototypes.items():
            if self.distance_metric == 'euclidean':
                dist = torch.norm(query_features - prototype.unsqueeze(0), dim=1)
            else:
                query_norm = F.normalize(query_features, p=2, dim=1)
                prototype_norm = F.normalize(prototype.unsqueeze(0), p=2, dim=1)
                similarity = F.cosine_similarity(query_norm, prototype_norm, dim=1)
                dist = 1 - similarity
            distances[combo] = dist
        return distances
    
    def aggregate_to_labels(self, distances, episode_classes, aggregation='max'):
        n_query = list(distances.values())[0].shape[0]
        n_classes = len(episode_classes)
        device = list(distances.values())[0].device
        
        if aggregation == 'max':
            label_scores = torch.full((n_query, n_classes), float('inf'), device=device)
        else:
            label_scores = torch.zeros((n_query, n_classes), device=device)
        
        temp_scores = []
        for combo, dist in distances.items():
            for cat_id in combo:
                if cat_id in episode_classes:
                    class_idx = episode_classes.index(cat_id)
                    temp_scores.append((class_idx, dist))
        
        if aggregation == 'max':
            for class_idx, dist in temp_scores:
                label_scores[:, class_idx] = torch.where(
                    dist < label_scores[:, class_idx],
                    dist,
                    label_scores[:, class_idx]
                )
        label_logits = -label_scores + self.bias
        return label_logits
    
    def forward(self, episode):
        support_features = self.feature_extractor(episode['support_images'])
        query_features = self.feature_extractor(episode['query_images'])
        
        prototypes = self.compute_prototypes(support_features, episode['support_labels'])
        distances = self.compute_distances(query_features, prototypes)
        predictions = self.aggregate_to_labels(distances, episode['episode_classes'])
        
        return predictions

# ==================== CoMC Classifier (from Step 4) ====================
class CoMCClassifier(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super(CoMCClassifier, self).__init__()
        self.classifier = nn.Linear(embedding_dim, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
    
    def forward(self, x):
        return self.classifier(x)

print("All model classes defined successfully!")

## 5.6 Define CoMC Inference and Episode Sampling


In [ ]:
def build_class_text_embeddings(class_names, clip_model, template, device):
    """Encode one CLIP text prompt for each class."""
    text_descriptions = [template.format(name) for name in class_names]
    text_tokens = clip.tokenize(text_descriptions).to(device)
    
    with torch.no_grad():
        text_features = clip_model.encode_text(text_tokens)
    
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features.float()

def cross_modal_mapping(image_embeddings, class_text_embeddings, temperature=0.07):
    """Map image embeddings into the CLIP text embedding space."""
    if image_embeddings.dim() == 2:
        image_embeddings = image_embeddings.unsqueeze(1)
        squeeze_output = True
    else:
        squeeze_output = False
    
    similarities = torch.einsum('bps,cs->bpc', image_embeddings, class_text_embeddings) / temperature
    weights = F.softmax(similarities, dim=-1)
    mapped_embeddings = torch.einsum('bpc,cs->bps', weights, class_text_embeddings)
    
    if squeeze_output:
        mapped_embeddings = mapped_embeddings.squeeze(1)
    
    return mapped_embeddings

def comc_inference(clip_model, comc_classifier, query_images_comc, 
                   class_text_embeddings, alpha=0.7, temperature=0.07, device=DEVICE):
    """Generate class probabilities with the CoMC component."""
    query_images_comc = query_images_comc.to(device)
    
    with torch.no_grad():
        image_features = clip_model.encode_image(query_images_comc)
        image_features = image_features.float()
        global_features = image_features / image_features.norm(dim=-1, keepdim=True)
        batch_size = global_features.shape[0]
        local_features = global_features.unsqueeze(1)
    
    mapped_global = cross_modal_mapping(global_features, class_text_embeddings, temperature)
    fused_global = alpha * global_features + (1 - alpha) * mapped_global
    
    global_logits = comc_classifier(fused_global)
    global_probs = torch.sigmoid(global_logits)
    
    mapped_local = cross_modal_mapping(local_features, class_text_embeddings, temperature)
    fused_local = alpha * local_features + (1 - alpha) * mapped_local
    
    batch_size, num_patches, embedding_dim = fused_local.shape
    fused_local_flat = fused_local.view(batch_size * num_patches, embedding_dim)
    
    local_logits_flat = comc_classifier(fused_local_flat)
    local_probs_flat = torch.sigmoid(local_logits_flat)
    local_probs = local_probs_flat.view(batch_size, num_patches, -1)
    aggregated_local_probs, _ = local_probs.max(dim=1)
    
    final_probs = 0.5 * (global_probs + aggregated_local_probs)
    return final_probs

class EpisodeSampler:
    def __init__(self, dataset, n_way, k_shot, n_query=4, seed=42):
        self.dataset = dataset
        self.n_way = n_way
        self.k_shot = k_shot
        self.n_query = n_query
        self.seed = seed
        self.rng = random.Random(seed)
        
        self.cat_to_indices = defaultdict(list)
        for idx, sample in enumerate(dataset.samples):
            for cat_id in sample['category_ids']:
                self.cat_to_indices[cat_id].append(idx)
        
        self.image_to_cats = defaultdict(list)
        for idx, sample in enumerate(dataset.samples):
            for cat_id in sample['category_ids']:
                self.image_to_cats[idx].append(cat_id)
    
    def sample_fixed_classes(self, fixed_cat_ids):
        """Sample support and query examples from a fixed set of classes."""
        episode_cats = fixed_cat_ids
        support_samples = []
        query_samples = []
        
        used_support_indices = set()
        used_query_indices = set()
        
        for cat_id in episode_cats:
            cat_indices = self.cat_to_indices[cat_id]
            
            available_for_support = [idx for idx in cat_indices 
                                    if idx not in used_query_indices]
            available_for_query = [idx for idx in cat_indices 
                                 if idx not in used_support_indices]
            
            needed_support = self.k_shot
            needed_query = self.n_query
            
            if len(available_for_support) < needed_support or len(available_for_query) < needed_query:
                if len(cat_indices) < self.k_shot + self.n_query:
                    selected_indices = self.rng.choices(cat_indices, k=self.k_shot + self.n_query)
                else:
                    selected_indices = self.rng.sample(cat_indices, k=self.k_shot + self.n_query)
            else:
                support_indices = self.rng.sample(available_for_support, needed_support)
                available_for_query_filtered = [idx for idx in available_for_query 
                                              if idx not in support_indices]
                
                if len(available_for_query_filtered) < needed_query:
                    remaining = needed_query - len(available_for_query_filtered)
                    query_indices = available_for_query_filtered + self.rng.choices(
                        [idx for idx in cat_indices if idx not in available_for_query_filtered],
                        k=remaining
                    )
                else:
                    query_indices = self.rng.sample(available_for_query_filtered, needed_query)
                
                selected_indices = support_indices + query_indices
            
            support_indices = selected_indices[:self.k_shot]
            query_indices = selected_indices[self.k_shot:self.k_shot + self.n_query]
            
            used_support_indices.update(support_indices)
            used_query_indices.update(query_indices)
            
            support_samples.extend(support_indices)
            query_samples.extend(query_indices)
        
        support_data = [self.dataset[idx] for idx in support_samples]
        query_data = [self.dataset[idx] for idx in query_samples]
        
        support_images_lc = torch.stack([item['image_lc'] for item in support_data])
        support_images_comc = torch.stack([item['image_comc'] for item in support_data])
        query_images_lc = torch.stack([item['image_lc'] for item in query_data])
        query_images_comc = torch.stack([item['image_comc'] for item in query_data])
        
        support_labels = [item['labels'].tolist() for item in support_data]
        query_labels = [item['labels'].tolist() for item in query_data]
        
        return {
            'support_images_lc': support_images_lc,
            'support_images_comc': support_images_comc,
            'query_images_lc': query_images_lc,
            'query_images_comc': query_images_comc,
            'support_labels': support_labels,
            'query_labels': query_labels,
            'episode_classes': episode_cats
        }

print("CoMC functions and EpisodeSampler defined!")

## 5.7 Define the Ensemble Model


In [ ]:
class EnsembleModel:
    """Combine LC-Protonets and CoMC predictions with a configurable weight."""
    
    def __init__(self, lc_protonets, clip_model, comc_classifier, 
                 class_text_embeddings, alpha=0.5, comc_alpha=0.7, 
                 comc_temperature=0.07, device=DEVICE):
        self.lc_protonets = lc_protonets
        self.clip_model = clip_model
        self.comc_classifier = comc_classifier
        self.class_text_embeddings = class_text_embeddings
        self.alpha = alpha
        self.comc_alpha = comc_alpha
        self.comc_temperature = comc_temperature
        self.device = device
        
        print(f"EnsembleModel initialized:")
        print(f"  Fusion alpha: {alpha}")
        print(f"  CoMC alpha: {comc_alpha}")
        print(f"  CoMC temperature: {comc_temperature}")
    
    @torch.no_grad()
    def predict(self, episode, return_individual=False):
        """Return predictions from both components and their weighted ensemble."""
        self.lc_protonets.eval()
        episode_lc = {
            'support_images': episode['support_images_lc'].to(self.device),
            'query_images': episode['query_images_lc'].to(self.device),
            'support_labels': episode['support_labels'],
            'query_labels': episode['query_labels'],
            'episode_classes': episode['episode_classes']
        }
        lc_logits = self.lc_protonets(episode_lc)
        p_proto = torch.sigmoid(lc_logits)
        
        self.comc_classifier.eval()
        query_images_comc = episode['query_images_comc']
        p_comc = comc_inference(
            self.clip_model,
            self.comc_classifier,
            query_images_comc,
            self.class_text_embeddings,
            alpha=self.comc_alpha,
            temperature=self.comc_temperature,
            device=self.device
        )
        
        p_final = self.alpha * p_proto + (1 - self.alpha) * p_comc
        
        if return_individual:
            individual_dict = {
                'p_proto': p_proto.cpu(),
                'p_comc': p_comc.cpu()
            }
            return p_final.cpu(), individual_dict
        else:
            return p_final.cpu()

print("EnsembleModel class defined!")

## 5.8 Define Evaluation Metrics


In [ ]:
def compute_multilabel_metrics(predictions, targets, verbose=False):
    """Compute macro- and micro-averaged AP and F1 scores."""
    predictions_np = predictions.numpy() if torch.is_tensor(predictions) else predictions
    targets_np = targets.numpy() if torch.is_tensor(targets) else targets
    
    try:
        map_macro = average_precision_score(targets_np, predictions_np, average='macro')
        map_micro = average_precision_score(targets_np, predictions_np, average='micro')
    except Exception as e:
        print(f"Error computing AP: {e}")
        map_macro = 0.0
        map_micro = 0.0
    
    pred_labels = (predictions_np > 0.5).astype(int)
    
    try:
        f1_macro = f1_score(targets_np, pred_labels, average='macro', zero_division=0)
        f1_micro = f1_score(targets_np, pred_labels, average='micro', zero_division=0)
    except Exception as e:
        print(f"Error computing F1: {e}")
        f1_macro = 0.0
        f1_micro = 0.0
    
    accuracy = (pred_labels == targets_np).mean()
    
    if verbose:
        print(f"\nDetailed Metrics:")
        print(f"  mAP (macro): {map_macro:.4f}")
        print(f"  mAP (micro): {map_micro:.4f}")
        print(f"  F1 (macro): {f1_macro:.4f}")
        print(f"  F1 (micro): {f1_micro:.4f}")
        print(f"  Accuracy: {accuracy:.4f}")
    
    return {
        'map_macro': map_macro,
        'map_micro': map_micro,
        'f1_macro': f1_macro,
        'f1_micro': f1_micro,
        'accuracy': accuracy
    }

print("Evaluation metrics function defined!")

## 5.9 Load Models and Prepare Data


In [ ]:
lc_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

_, clip_preprocess = clip.load(CLIP_MODEL, device=DEVICE)

print("Creating dataset...")
dataset = DualPreprocessCOCODataset(
    samples=novel_samples,
    train_image_dir=TRAIN2014_DIR,
    val_image_dir=VAL2014_DIR,
    lc_transform=lc_preprocess,
    comc_transform=clip_preprocess,
    cat_id_to_name=cat_id_to_name
)
print(f"Dataset created: {len(dataset)} samples")

print("\nLoading LC-Protonets...")
lc_feature_extractor = FeatureExtractor(
    pretrained=True, feature_dim=2048, finetune=True
).to(DEVICE)
lc_protonets = LCProtonets(
    feature_extractor=lc_feature_extractor,
    distance_metric='euclidean',
    init_bias=14.7
).to(DEVICE)
lc_protonets.load_state_dict(torch.load(LC_PROTONETS_CKPT, map_location=DEVICE))
lc_protonets.eval()
print(f"LC-Protonets loaded!")

print("\nLoading CoMC classifier...")
clip_model, _ = clip.load(CLIP_MODEL, device=DEVICE)
clip_model.eval()

comc_checkpoint = torch.load(COMC_CLASSIFIER_CKPT, map_location=DEVICE)
comc_class_names = comc_checkpoint['class_names']

if comc_class_names == novel_class_names:
    print("Class names aligned perfectly!")
else:
    print("WARNING: Class names mismatch!")

comc_classifier = CoMCClassifier(
    embedding_dim=512, num_classes=len(novel_class_names)
).to(DEVICE)
comc_classifier.load_state_dict(comc_checkpoint['classifier_state_dict'])
comc_classifier.eval()
print(f"CoMC classifier loaded!")

print("\nBuilding class text embeddings...")
TEXT_TEMPLATE = "a photo of a {}"
class_text_embeddings = build_class_text_embeddings(
    novel_class_names, clip_model, TEXT_TEMPLATE, DEVICE
)
print(f"Class text embeddings ready: {class_text_embeddings.shape}")

sampler = EpisodeSampler(dataset, N_WAY, K_SHOT, N_QUERY, seed=EXPERIMENT_SEED)
print(f"Episode sampler ready: {N_WAY}-way, {K_SHOT}-shot, seed={EXPERIMENT_SEED}")

ensemble_model = EnsembleModel(
    lc_protonets=lc_protonets,
    clip_model=clip_model,
    comc_classifier=comc_classifier,
    class_text_embeddings=class_text_embeddings,
    alpha=ALPHA,
    comc_alpha=COMC_ALPHA,
    comc_temperature=COMC_TEMPERATURE,
    device=DEVICE
)

print("\n" + "="*70)
print("ALL MODELS LOADED AND READY!")
print("="*70)

## 5.10 Run the Ensemble Evaluation


In [ ]:
def evaluate_ensemble(ensemble_model, sampler, novel_cat_ids, n_episodes, device=DEVICE):
    """Evaluate LC-Protonets, CoMC, and their ensemble over few-shot episodes."""
    print(f"\n{'='*70}")
    print(f"RUNNING ENSEMBLE EVALUATION")
    print(f"{'='*70}")
    print(f"\nEvaluating {n_episodes} episodes...")
    
    all_p_proto = []
    all_p_comc = []
    all_p_final = []
    all_targets = []
    
    for episode_idx in tqdm(range(n_episodes), desc="Evaluating"):
        episode = sampler.sample_fixed_classes(novel_cat_ids)
        p_final, individual = ensemble_model.predict(episode, return_individual=True)
        
        p_proto = individual['p_proto']
        p_comc = individual['p_comc']
        
        n_query = p_final.shape[0]
        n_classes = len(episode['episode_classes'])
        target_tensor = torch.zeros(n_query, n_classes)
        
        for i, labels in enumerate(episode['query_labels']):
            for label in labels:
                if label in episode['episode_classes']:
                    class_idx = episode['episode_classes'].index(label)
                    target_tensor[i, class_idx] = 1
        
        all_p_proto.append(p_proto)
        all_p_comc.append(p_comc)
        all_p_final.append(p_final)
        all_targets.append(target_tensor)

        completed_episode = episode_idx + 1
        if completed_episode == 1 or completed_episode % 5 == 0 or completed_episode == n_episodes:
            update_run_progress(
                'Test episode evaluation',
                completed_episode,
                n_episodes
            )
    
    all_p_proto = torch.cat(all_p_proto, dim=0)
    all_p_comc = torch.cat(all_p_comc, dim=0)
    all_p_final = torch.cat(all_p_final, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    print(f"\nEvaluation completed!")
    print(f"Total samples: {all_targets.shape[0]}")
    
    print(f"\n{'='*70}")
    print(f"COMPUTING METRICS")
    print(f"{'='*70}")
    
    print("\n1. LC-Protonets (alone):")
    metrics_proto = compute_multilabel_metrics(all_p_proto, all_targets, verbose=True)
    
    print("\n2. CoMC (alone):")
    metrics_comc = compute_multilabel_metrics(all_p_comc, all_targets, verbose=True)
    
    print("\n3. Ensemble (alpha={}):".format(ensemble_model.alpha))
    metrics_ensemble = compute_multilabel_metrics(all_p_final, all_targets, verbose=True)
    
    return {
        'lc_protonets': metrics_proto,
        'comc': metrics_comc,
        'ensemble': metrics_ensemble,
        'predictions': {
            'p_proto': all_p_proto,
            'p_comc': all_p_comc,
            'p_final': all_p_final,
            'targets': all_targets
        }
    }

results = evaluate_ensemble(
    ensemble_model=ensemble_model,
    sampler=sampler,
    novel_cat_ids=novel_cat_ids,
    n_episodes=N_EPISODES,
    device=DEVICE
)

print(f"\n{'='*70}")
print(f"EVALUATION SUMMARY")
print(f"{'='*70}")
print(f"\n{'Model':<15} {'mAP(mac)':<10} {'mAP(mic)':<10} {'F1(mac)':<10} {'F1(mic)':<10}")
print("-" * 70)
for key, name in [('lc_protonets', 'LC-Protonets'), ('comc', 'CoMC'), ('ensemble', 'Ensemble')]:
    m = results[key]
    print(f"{name:<15} {m['map_macro']:<10.4f} {m['map_micro']:<10.4f} {m['f1_macro']:<10.4f} {m['f1_micro']:<10.4f}")

print(f"\nEnsemble vs LC-Protonets: {results['ensemble']['map_macro'] - results['lc_protonets']['map_macro']:+.4f}")
print(f"Ensemble vs CoMC: {results['ensemble']['map_macro'] - results['comc']['map_macro']:+.4f}")
print("="*70)

## 5.11 Visualize Model Comparisons


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'Ensemble Evaluation Results (alpha={ALPHA}, {N_EPISODES} episodes)',
             fontsize=16, fontweight='bold')

models = ['LC-Protonets', 'CoMC', 'Ensemble']
map_macro = [results['lc_protonets']['map_macro'],
             results['comc']['map_macro'],
             results['ensemble']['map_macro']]
f1_macro = [results['lc_protonets']['f1_macro'],
            results['comc']['f1_macro'],
            results['ensemble']['f1_macro']]

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

ax1 = axes[0, 0]
x = np.arange(len(models))
bars = ax1.bar(x, map_macro, color=colors[2], alpha=0.8)
ax1.set_ylabel('Average Precision', fontweight='bold')
ax1.set_title('mAP Comparison (Macro)', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.set_ylim([0, 1])
ax1.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, map_macro):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2 = axes[0, 1]
bars = ax2.bar(x, f1_macro, color=colors[2], alpha=0.8)
ax2.set_ylabel('F1 Score', fontweight='bold')
ax2.set_title('F1 Score Comparison (Macro)', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(models)
ax2.set_ylim([0, 1])
ax2.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, f1_macro):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax3 = axes[1, 0]
p_final_flat = results['predictions']['p_final'].numpy().flatten()
ax3.hist(p_final_flat, bins=50, color=colors[2], alpha=0.7, edgecolor='black')
ax3.set_xlabel('Predicted Probability', fontweight='bold')
ax3.set_ylabel('Frequency', fontweight='bold')
ax3.set_title('Ensemble Probability Distribution', fontweight='bold')
ax3.axvline(x=0.5, color='red', linestyle='--', linewidth=2)
ax3.grid(alpha=0.3)

ax4 = axes[1, 1]
ax4.axis('off')
table_data = [
    ['Model', 'mAP (Macro)', 'F1 (Macro)'],
    ['LC-Protonets', f"{results['lc_protonets']['map_macro']:.4f}",
     f"{results['lc_protonets']['f1_macro']:.4f}"],
    ['CoMC', f"{results['comc']['map_macro']:.4f}",
     f"{results['comc']['f1_macro']:.4f}"],
    ['Ensemble', f"{results['ensemble']['map_macro']:.4f}",
     f"{results['ensemble']['f1_macro']:.4f}"]
]
table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                 colWidths=[0.3, 0.3, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)
ax4.set_title('Metrics Summary', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ensemble_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"Visualization saved!")

## 5.12 Save Evaluation Results


In [ ]:
import json
results_file = os.path.join(OUTPUT_DIR, 'ensemble_evaluation_results.json')

results_to_save = {
    'configuration': {
        'n_episodes': N_EPISODES,
        'n_way': N_WAY,
        'k_shot': K_SHOT,
        'n_query': N_QUERY,
        'ensemble_alpha': ALPHA,
        'experiment_seed': EXPERIMENT_SEED,
        'episode_seed': EXPERIMENT_SEED
    },
    'metrics': {
        'lc_protonets': results['lc_protonets'],
        'comc': results['comc'],
        'ensemble': results['ensemble']
    },
    'improvements': {
        'ensemble_vs_lc_protonets': float(results['ensemble']['map_macro'] - results['lc_protonets']['map_macro']),
        'ensemble_vs_comc': float(results['ensemble']['map_macro'] - results['comc']['map_macro'])
    },
    'class_names': novel_class_names
}

with open(results_file, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f"Results saved to: {results_file}")
print("\nFinal mAP (macro):")
print(f"  LC-Protonets: {results['lc_protonets']['map_macro']:.4f}")
print(f"  CoMC: {results['comc']['map_macro']:.4f}")
print(f"  Ensemble: {results['ensemble']['map_macro']:.4f}")

## Summary

This notebook evaluates an ensemble of LC-Protonets and CoMC.

### Key Features
1. Load both models and align their class order.
2. Apply separate preprocessing pipelines for ResNet and CLIP.
3. Combine model predictions with a configurable weight.
4. Compute metrics and visualize results.
5. Save evaluation outputs.

### Metrics
- mAP (macro and micro)
- F1 score (macro and micro)
- Accuracy

### Output Files
Evaluation artifacts are saved under the configured output directory.
